# Feature Extraction

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, LSTM, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- Load data and create windows (if not already loaded) ---
# Check if windows and window_labels already exist
if 'windows' not in globals() or 'window_labels' not in globals():
    # Load data from previous notebook
    data_dir = Path('../data/raw/CAPPIMU/data')
    subject_id = 1
    
    def read_subject_trial(subject_id, data_dir=data_dir):
        subject_path = data_dir / f'subject_{subject_id}'
        if not subject_path.exists():
            raise ValueError(f"Subject {subject_id} data not found")
        
        csv_path = subject_path / 'insole.csv'
        if csv_path.exists():
            return pd.read_csv(csv_path)
        else:
            raise FileNotFoundError(f"Insole data not found: {csv_path}")
    
    # Load insole data
    data = read_subject_trial(subject_id)
    
    # Separate sensor values (exclude timestamp and label columns)
    sensor_cols = [col for col in data.columns if col not in ['time', 'label']]
    sensor_data = data[sensor_cols].values  # shape: (num_samples, num_channels)
    
    # Convert string labels to integers
    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(data['label'].values)
    print(f"Label mapping: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")
    
    # Define window parameters
    fs = 100  # sensor frequency in Hz
    window_sec = 1.0  # 1-second window
    window_size = int(fs * window_sec)  # samples per window
    stride = int(window_size * 0.5)  # 50% overlap
    
    # Windowing function
    def create_windows_multi_channel(sensor_data, labels=None, window_size=100, stride=50):
        windows = []
        window_labels = []
        for start in range(0, len(sensor_data) - window_size + 1, stride):
            end = start + window_size
            windows.append(sensor_data[start:end, :])  # multi-channel window
            if labels is not None:
                # Assign window label: majority label in the window
                window_labels.append(np.bincount(labels[start:end]).argmax())
        windows = np.array(windows)  # shape: (num_windows, window_size, num_channels)
        if labels is not None:
            window_labels = np.array(window_labels)
            return windows, window_labels
        return windows
    
    # Create windows
    windows, window_labels = create_windows_multi_channel(sensor_data, labels, window_size, stride)
    print(f"Windows shape: {windows.shape}")
    print(f"Window labels shape: {window_labels.shape}")

# One-hot encode labels for 21 classes
y_onehot = to_categorical(window_labels, num_classes=21)

# Optional: train-test split
X_train, X_test, y_train, y_test = train_test_split(windows, y_onehot, test_size=0.2, random_state=42, stratify=y_onehot)

# Optional: standardize input per channel
num_channels = X_train.shape[2]
for ch in range(num_channels):
    scaler = StandardScaler()
    X_train[:, :, ch] = scaler.fit_transform(X_train[:, :, ch])
    X_test[:, :, ch] = scaler.transform(X_test[:, :, ch])

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")



Label mapping: {'brushing_teeth': 0, 'cut': 1, 'cycle': 2, 'drink': 3, 'eat': 4, 'fall': 5, 'folding_clothes': 6, 'hang_out_clothes': 7, 'ironing': 8, 'mop': 9, 'play_phone': 10, 'run': 11, 'sweep': 12, 'use_computer': 13, 'walk': 14, 'wash_dish': 15, 'wash_face': 16, 'wash_window': 17, 'watch_tv': 18, 'wc': 19, 'write': 20}
Windows shape: (4072, 100, 16)
Window labels shape: (4072,)

Training set shape: (3257, 100, 16)
Test set shape: (815, 100, 16)
Training labels shape: (3257, 21)
Test labels shape: (815, 21)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization

model = Sequential()

# CNN feature extractor
model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(100, 16)))
model.add(BatchNormalization())
model.add(Conv1D(64, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# Temporal modeling
model.add(LSTM(128))
model.add(Dropout(0.3))

# Classifier
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(21, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## Model Input Preparation

- Windowed sensor data shape: (4072, 100, 16)
  - 100 time steps per window (1 s at 100 Hz)
  - 16 sensor channels (FSR + IMU)
- Labels encoded into 21 activity classes using one-hot encoding
- Dataset split into training (80%) and test (20%) sets
- Training data standardized per sensor channel using z-score normalization
- Overlapping windows used (50% overlap)

This processed data is used as input to a CNN–LSTM model for activity classification.

In [9]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)


Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 41s 445ms/step - accuracy: 0.3666 - loss: 2.2093 - val_accuracy: 0.5752 - val_loss: 1.5978
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 14s 306ms/step - accuracy: 0.6395 - loss: 1.1985 - val_accuracy: 0.7423 - val_loss: 0.9399
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 13s 316ms/step - accuracy: 0.7420 - loss: 0.8357 - val_accuracy: 0.7408 - val_loss: 0.7870
Epoch 4/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 6s 146ms/step - accuracy: 0.8004 - loss: 0.6864 - val_accuracy: 0.7975 - val_loss: 0.6154
Epoch 5/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - accuracy: 0.8177 - loss: 0.6065 - val_accuracy: 0.8681 - val_loss: 0.4370
Epoch 6/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.8683 - loss: 0.4524 - val_accuracy: 0.8834 - val_loss: 0.3423
Epoch 7/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 90ms/step - accuracy: 0.8864 - loss: 0.3892 - val_accuracy: 0.9187 - val_loss: 0.2786
Epoch 8/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - accuracy: 0.8856 - loss: 0.3877 - val_accuracy: 

In [10]:
import numpy as np

# Predict probabilities
y_pred_probs = model.predict(X_test)

# True labels (class index)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Fall = class 5
FALL_CLASS = 5

y_true_fall = (y_true == FALL_CLASS).astype(int)
y_pred_fall = (y_pred == FALL_CLASS).astype(int)


26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

print("Fall vs Non-Fall Classification Report:")
print(classification_report(
    y_true_fall,
    y_pred_fall,
    target_names=["Non-Fall", "Fall"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_true_fall, y_pred_fall))


Fall vs Non-Fall Classification Report:
              precision    recall  f1-score   support

    Non-Fall       0.99      1.00      0.99       782
        Fall       0.92      0.70      0.79        33

    accuracy                           0.99       815
   macro avg       0.95      0.85      0.89       815
weighted avg       0.98      0.99      0.98       815

Confusion Matrix:
[[780   2]
 [ 10  23]]


## Initial Fall Detection Test (Multi-Class Model)

The model was initially trained to classify 21 activities, including falls (class 5). After training, fall detection was evaluated as a binary problem (fall vs non-fall) to assess its real-world effectiveness.

### Results
- **Precision for falls:** 0.92  
- **Recall for falls:** 0.70  
- **Support:** 33 fall windows in the test set  
- **Confusion matrix:**

### Interpretation
- While overall accuracy was high (0.99), this is **misleading** due to class imbalance.  
- **False negatives (10 missed falls)** are a significant issue, as falls are safety-critical events.  
- This initial test demonstrates that a multi-class activity model is **not optimal for fall detection**, highlighting the need for either:
  1. Threshold tuning on fall probability, or  
  2. Training a dedicated binary fall detection model.

In [12]:
import numpy as np
from sklearn.metrics import classification_report

fall_probs = y_pred_probs[:, 5]  # fall class

for thresh in [0.1, 0.2, 0.3, 0.4]:
    y_pred_fall = (fall_probs >= thresh).astype(int)
    print(f"\nThreshold = {thresh}")
    print(classification_report(
        y_true_fall,
        y_pred_fall,
        target_names=["Non-Fall", "Fall"]
    ))



Threshold = 0.1
              precision    recall  f1-score   support

    Non-Fall       0.99      0.99      0.99       782
        Fall       0.86      0.73      0.79        33

    accuracy                           0.98       815
   macro avg       0.92      0.86      0.89       815
weighted avg       0.98      0.98      0.98       815


Threshold = 0.2
              precision    recall  f1-score   support

    Non-Fall       0.99      1.00      0.99       782
        Fall       0.88      0.70      0.78        33

    accuracy                           0.98       815
   macro avg       0.94      0.85      0.89       815
weighted avg       0.98      0.98      0.98       815


Threshold = 0.3
              precision    recall  f1-score   support

    Non-Fall       0.99      1.00      0.99       782
        Fall       0.92      0.70      0.79        33

    accuracy                           0.99       815
   macro avg       0.95      0.85      0.89       815
weighted avg       0.98

In [13]:
for thresh in np.arange(0.05, 0.31, 0.05):
    y_pred_fall = (fall_probs >= thresh).astype(int)
    print(f"\nThreshold = {thresh}")
    print(classification_report(y_true_fall, y_pred_fall, target_names=["Non-Fall", "Fall"]))



Threshold = 0.05
              precision    recall  f1-score   support

    Non-Fall       0.99      0.99      0.99       782
        Fall       0.83      0.76      0.79        33

    accuracy                           0.98       815
   macro avg       0.91      0.88      0.89       815
weighted avg       0.98      0.98      0.98       815


Threshold = 0.1
              precision    recall  f1-score   support

    Non-Fall       0.99      0.99      0.99       782
        Fall       0.86      0.73      0.79        33

    accuracy                           0.98       815
   macro avg       0.92      0.86      0.89       815
weighted avg       0.98      0.98      0.98       815


Threshold = 0.15000000000000002
              precision    recall  f1-score   support

    Non-Fall       0.99      1.00      0.99       782
        Fall       0.88      0.70      0.78        33

    accuracy                           0.98       815
   macro avg       0.94      0.85      0.89       815
weight

In [14]:
from sklearn.utils.class_weight import compute_class_weight

# Original window_labels: shape (4072,)
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(window_labels),
    y=window_labels
)
class_weights_dict = dict(enumerate(class_weights))

# During model.fit()
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    class_weight=class_weights_dict  # <-- critical
)


Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - accuracy: 0.9843 - loss: 0.0510 - val_accuracy: 0.9617 - val_loss: 0.1573
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.9843 - loss: 0.0493 - val_accuracy: 0.9632 - val_loss: 0.1315
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.9827 - loss: 0.0668 - val_accuracy: 0.9509 - val_loss: 0.1940
Epoch 4/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.9685 - loss: 0.0923 - val_accuracy: 0.9509 - val_loss: 0.2212
Epoch 5/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.9731 - loss: 0.0772 - val_accuracy: 0.9678 - val_loss: 0.1376
Epoch 6/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.9727 - loss: 0.0850 - val_accuracy: 0.9571 - val_loss: 0.1528
Epoch 7/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.9743 - loss: 0.0802 - val_accuracy: 0.9417 - val_loss: 0.2038
Epoch 8/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.9785 - loss: 0.0887 - val_accuracy: 0.9632 - v

In [17]:
# Predict probabilities for test set
y_pred_probs = model.predict(X_test)

# Convert to class indices (multi-class)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# True labels (class index)
y_true_classes = np.argmax(y_test, axis=1)


26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step


In [18]:
FALL_CLASS = 5

y_true_fall = (y_true_classes == FALL_CLASS).astype(int)
y_pred_fall = (y_pred_classes == FALL_CLASS).astype(int)


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

print("Fall vs Non-Fall Classification Report:")
print(classification_report(
    y_true_fall,
    y_pred_fall,
    target_names=["Non-Fall", "Fall"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_true_fall, y_pred_fall))


Fall vs Non-Fall Classification Report:
              precision    recall  f1-score   support

    Non-Fall       0.99      1.00      0.99       782
        Fall       0.96      0.76      0.85        33

    accuracy                           0.99       815
   macro avg       0.98      0.88      0.92       815
weighted avg       0.99      0.99      0.99       815

Confusion Matrix:
[[781   1]
 [  8  25]]


## Fall Detection After Weighted Loss

After incorporating class weights to address class imbalance, the model was re-evaluated for fall detection as a binary problem (fall vs non-fall).

### Results:

- **Precision (Fall):** 0.96  
- **Recall (Fall):** 0.76  
- **F1-score (Fall):** 0.85  
- **Support:** 33 fall windows in the test set  

**Confusion Matrix:**



### Interpretation:

- Recall for falls improved from 0.70 → 0.76 due to weighted loss, meaning **fewer missed falls**.  
- Precision remains high (0.96), indicating most predicted falls are correct.  
- False negatives reduced (8 missed falls vs 10 previously), which is critical for safety applications.  
- Overall accuracy remains high (0.99), but the key improvement is in the **fall class performance**.  

**Conclusion:** Weighted loss effectively mitigates class imbalance and enhances fall detection, prioritizing recall without significantly affecting precision.
